# CineSen – Train ABSA trên Kaggle (GPU)

Train model **Aspect-Based Sentiment Analysis** (RoBERTa + multi-label head) trên Kaggle với GPU để tăng tốc.

**Cách dùng:**  
1. Tạo Dataset trên Kaggle, upload file `labeled_absa_auto.jsonl`.  
2. Tạo Notebook mới, bật **GPU** (Settings → Accelerator → GPU).  
3. Add dataset vào notebook.  
4. Sửa `INPUT_DATA_PATH` cho đúng đường dẫn file trong dataset (vd: `/kaggle/input/your-dataset/labeled_absa_auto.jsonl`).  
5. Run All. Artifact ở `/kaggle/working/absa_artifact` → tải về hoặc tạo Output dataset.

In [ ]:
# Cấu hình (sửa INPUT_DATA_PATH theo dataset của bạn trên Kaggle)
import os
import json
from pathlib import Path

# Đường dẫn file labeled JSONL trong Kaggle input (sau khi Add data)
INPUT_DATA_PATH = "/kaggle/input/cinesen-absa-labeled/labeled_absa_auto.jsonl"

# Tự tìm file .jsonl trong /kaggle/input nếu đường dẫn trên không tồn tại
if not os.path.isfile(INPUT_DATA_PATH) and os.path.isdir("/kaggle/input"):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".jsonl") and "label" in f.lower():
                INPUT_DATA_PATH = os.path.join(root, f)
                print("Đã chọn file:", INPUT_DATA_PATH)
                break
        else:
            continue
        break

OUTPUT_ARTIFACT_DIR = "/kaggle/working/absa_artifact"
MODEL_NAME = "roberta-base"
BATCH_SIZE = 16   # GPU lớn (H100) có thể tăng 32–64
EPOCHS = 3
LR = 2e-5
MAX_LENGTH = 128

assert os.path.isfile(INPUT_DATA_PATH), f"Không tìm thấy data: {INPUT_DATA_PATH}. Hãy Add dataset và sửa INPUT_DATA_PATH."
print("Data:", INPUT_DATA_PATH)
print("GPU:", os.environ.get("CUDA_VISIBLE_DEVICES", "not set"))

In [ ]:
# Schema ABSA (7 aspect x 3 sentiment = 21 nhãn)
ASPECTS = ["script", "acting", "visuals", "music", "pacing", "direction", "overall"]
SENTIMENTS = ["negative", "neutral", "positive"]
NUM_LABELS = len(ASPECTS) * len(SENTIMENTS)

def get_label_index(aspect: str, sentiment: str) -> int:
    a = ASPECTS.index(aspect) if aspect in ASPECTS else -1
    s = SENTIMENTS.index(sentiment) if sentiment in SENTIMENTS else -1
    if a < 0 or s < 0:
        raise ValueError(f"Unknown aspect={aspect!r} or sentiment={sentiment!r}")
    return a * len(SENTIMENTS) + s

def index_to_aspect_sentiment(idx: int) -> tuple:
    if idx < 0 or idx >= NUM_LABELS:
        raise ValueError(f"Label index out of range: {idx}")
    a = idx // len(SENTIMENTS)
    s = idx % len(SENTIMENTS)
    return ASPECTS[a], SENTIMENTS[s]

def build_label_map():
    return [
        {"index": i, "aspect": ASPECTS[i // len(SENTIMENTS)], "sentiment": SENTIMENTS[i % len(SENTIMENTS)]}
        for i in range(NUM_LABELS)
    ]
print("NUM_LABELS:", NUM_LABELS)

In [ ]:
# Dataset
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
import torch.nn as nn

class AbsaDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=128):
        self.samples = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                text = (obj.get("text") or "").strip()
                labels_raw = obj.get("labels") or []
                if not text:
                    continue
                label_vec = [0.0] * NUM_LABELS
                for item in labels_raw:
                    a, s = item.get("aspect"), item.get("sentiment")
                    if a and s:
                        try:
                            label_vec[get_label_index(a, s)] = 1.0
                        except ValueError:
                            pass
                self.samples.append((text, label_vec))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        text, label_vec = self.samples[i]
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label_vec, dtype=torch.float32),
        }

In [ ]:
# Model: RoBERTa + linear head
class AbsaClassifier(nn.Module):
    def __init__(self, model_name: str, num_labels: int = NUM_LABELS):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.head = nn.Linear(hidden, num_labels)
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls)

In [ ]:
# Train
from datetime import datetime, timezone

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = AbsaDataset(INPUT_DATA_PATH, tokenizer, max_length=MAX_LENGTH)
print("Số mẫu:", len(dataset))
if len(dataset) == 0:
    raise ValueError("Không có mẫu nào trong file. Kiểm tra INPUT_DATA_PATH và format JSONL.")

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True if device.type == "cuda" else False,
)
model = AbsaClassifier(MODEL_NAME, NUM_LABELS).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
# Training loop
model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        opt.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{EPOCHS}  loss: {total_loss / len(loader):.4f}")
print("Train xong.")

In [ ]:
# Lưu artifact (dùng được với load_absa_artifact trong project)
Path(OUTPUT_ARTIFACT_DIR).mkdir(parents=True, exist_ok=True)
model.backbone.save_pretrained(OUTPUT_ARTIFACT_DIR)
tokenizer.save_pretrained(OUTPUT_ARTIFACT_DIR)
torch.save(model.head.state_dict(), Path(OUTPUT_ARTIFACT_DIR) / "head.pt")

schema = {"aspects": build_label_map(), "num_labels": NUM_LABELS}
(Path(OUTPUT_ARTIFACT_DIR) / "schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
metadata = {
    "model_name": MODEL_NAME,
    "dataset_size": len(dataset),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(Path(OUTPUT_ARTIFACT_DIR) / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Đã lưu artifact tại:", OUTPUT_ARTIFACT_DIR)
print("Trong Kaggle: vào Output tab để tải file hoặc tạo Dataset từ Output.")

In [ ]:
# (Tuỳ chọn) Test inference nhanh
model.eval()
test_texts = ["The acting was brilliant but the script was weak.", "Visuals and music were stunning."]
with torch.no_grad():
    for text in test_texts:
        enc = tokenizer(text, max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors="pt")
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        logits = model(input_ids, attention_mask).squeeze(0)
        probs = torch.sigmoid(logits).cpu().numpy()
        print(f"Text: {text[:60]}...")
        for idx, p in enumerate(probs):
            if p >= 0.5:
                asp, sent = index_to_aspect_sentiment(idx)
                print(f"  {asp} -> {sent} ({p:.2f})")
        print()

In [ ]:
# Zip artifact để tải về từ Kaggle
import shutil
from pathlib import Path

zip_path = Path(OUTPUT_ARTIFACT_DIR).with_suffix(".zip")
if Path(OUTPUT_ARTIFACT_DIR).exists():
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", OUTPUT_ARTIFACT_DIR)
    print("Đã zip artifact:", zip_path)
    print("Trong Kaggle, mở tab Output và tải file này về.")
else:
    print("Chưa thấy thư mục artifact, hãy chạy cell train & save trước.")